# GF(8) Toric Code - Canonical Champion Search (Local)

Local runner for the canonical search. By default it uses OpenCL for distance evaluation and a CPU canonical-form fallback when CUDA/CuPy canonicalization is unavailable.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(ROOT)

## Parameters

In [ ]:
# Kernel choices:
#   "opencl"  - local OpenCL distance kernel from kernel.py
#   "cuda_bp" - CUDA bit-parallel distance kernel, if this machine has CUDA/CuPy
KERNEL = "opencl"

MAX_K = 11
PRUNE_MARGIN = 6
PRUNE_EVAL_MODE = "champion"  # "champion" is fast; "survivor" is exact but can stall at larger k

# Local fallback canonicalization is CPU-based and supports k <= 10.
# If KERNEL="cuda_bp", the CUDA canonical kernel can support up to k <= 16.
K_CANONICAL_MAX = 10

# None = choose from device size. Reduce this if memory pressure or UI stalls.
BATCH_SIZE_OVERRIDE = None

# Set True only for smoke tests on machines with CPU OpenCL but no GPU OpenCL.
ALLOW_CPU_OPENCL = False

## Device Setup

In [ ]:
from precompute import build_eval_matrix, bounding_cube_points

M = build_eval_matrix()
lattice = bounding_cube_points()

if KERNEL == "cuda_bp":
    from kernel_cuda_bp import init_cuda_oracles_bp
    oracles, sm_count = init_cuda_oracles_bp(M)
    n_devices = len(oracles)
    default_batch = sm_count * 1024 * n_devices
elif KERNEL == "opencl":
    import os
    os.environ.setdefault("PYOPENCL_CTX", "0")
    from precompute import init_opencl
    from kernel import DistanceOracle
    ctx, queue_cl, M_buf, _ = init_opencl(allow_cpu=ALLOW_CPU_OPENCL)
    oracle = DistanceOracle(ctx, queue_cl, M_buf)
    oracles = [oracle]
    n_devices = 1
    default_batch = max(1024, int(queue_cl.device.max_compute_units) * 512)
else:
    raise ValueError(f"Unknown KERNEL={KERNEL!r}")

BATCH_SIZE = BATCH_SIZE_OVERRIDE or default_batch
print(f"Kernel: {KERNEL}")
print(f"Devices: {n_devices}")
print(f"Batch size: {BATCH_SIZE:,}")

## Codetables Bounds

In [ ]:
from codetables import bounds_for_n
import pandas as pd

targets = bounds_for_n(49)
df_bounds = pd.DataFrame(
    [(k, d) for k, d in sorted(targets.items()) if k <= 20],
    columns=["k", "best_known_d"],
)
display(df_bounds)

## Run Search

In [ ]:
import time
from datetime import datetime
from pathlib import Path
from champion_search_canonical import canonical_champion_search

local_results = sorted(Path(".").glob("canon_local_*.json"))
results_file = local_results[-1] if local_results else None
if results_file is None:
    results_file = Path(f"canon_local_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    print(f"Starting fresh: {results_file}")
else:
    print(f"Resuming: {results_file}")

t0 = time.perf_counter()
canonical_champion_search(
    oracles=oracles,
    lattice=lattice,
    results_file=results_file,
    targets=targets,
    max_k=MAX_K,
    batch_size=BATCH_SIZE,
    prune_margin=PRUNE_MARGIN,
    resume=True,
    k_canonical_max=K_CANONICAL_MAX,
    prune_eval_mode=PRUNE_EVAL_MODE,
)
print(f"\nTotal time: {time.perf_counter() - t0:.1f}s")

## Results

In [ ]:
from champion_search_canonical import load_results

records = [
    r for r in load_results(results_file)
    if r.get("type") not in ("level_complete", "survivor")
    and "k" in r and "min_distance" in r
]

if records:
    df_res = pd.DataFrame(records)[["name", "k", "min_distance", "best_known_d", "new_record", "indices"]]
    df_res["gap"] = df_res["best_known_d"] - df_res["min_distance"]
    df_res.sort_values(["k", "min_distance"], ascending=[True, False], inplace=True)
    df_res.reset_index(drop=True, inplace=True)
    display(df_res)
else:
    print("No champion records yet.")

In [ ]:
level_recs = [r for r in load_results(results_file) if r.get("type") == "level_complete"]
if level_recs:
    cols = ["k", "n_canonical", "n_champions", "n_survivors", "prune_eval_mode", "eval_distance", "keep_distance"]
    display(pd.DataFrame(level_recs)[cols])
else:
    print("No completed levels yet.")